# EP18 — Continual Learning: Three Families
**COMPSCI 713 · S1 2025 Q9 · 3 marks**

Briefly explain the core principle behind each family and how it mitigates catastrophic forgetting:
- Replay methods
- Regularisation-based methods
- Architecture-based methods

*Note: EP08 covers replay in depth. This EP covers all 3 families for the 2025 exam.*

---

## Catastrophic Forgetting — The Problem

When a neural network trained on Task T1 is then trained on Task T2, gradient updates for T2 overwrite the weights encoding T1. Performance on T1 collapses.

Root cause: all tasks compete for the same shared parameter space, and new gradients push parameters away from configurations that were optimal for old tasks.

---

## Family 1 — Replay Methods

**Core principle:** Mix samples (real or generated) from past tasks into the current training batch, so the network receives gradient signal from old tasks simultaneously with new ones.

**How it mitigates forgetting:** Old-task gradients pull weights back toward old-task-optimal regions while new-task gradients push forward — the balance retains past knowledge.

**Variants:**
- **Experience Replay (ER):** store a buffer of raw old-task samples; sample from it during new training
- **Generative Replay:** train a generative model (GAN/VAE) alongside; use it to synthesise old-task data (avoids storing raw data — privacy-friendly)
- **DER++ (Dark Experience Replay):** store input + model's old output logits; replay with both label loss and distillation loss on stored logits

**Trade-off:** buffer size limits how much old-task knowledge can be retained; small buffers → some forgetting remains.

---

## Family 2 — Regularisation-Based Methods

**Core principle:** Add a penalty to the loss function that discourages large changes to parameters that were *important* for old tasks.

**How it mitigates forgetting:** Important parameters are effectively "pinned" — the model can still learn new tasks but is constrained from moving away from old-task-critical weights.

**Key example — EWC (Elastic Weight Consolidation):**
- After training on T1, compute the **Fisher information matrix** to estimate each weight's importance to T1
- When training on T2, add a penalty term: `λ Σ_i F_i (θ_i − θ*_i)²`
  - F_i = Fisher importance of weight i
  - θ*_i = optimal weight for T1
  - Large F_i → weight is heavily penalised for moving → effectively frozen

**Other examples:** SI (Synaptic Intelligence), online EWC, LwF (Learning without Forgetting — uses old model outputs as soft labels)

**Trade-off:** The regularisation is an approximation — with many sequential tasks, the constraints accumulate and eventually become too rigid ("capacity saturation").

---

## Family 3 — Architecture-Based Methods

**Core principle:** Dedicate separate model components (subnetworks, masks, or modules) to different tasks, so tasks never compete for the exact same parameters.

**How it mitigates forgetting:** Each task gets its own private parameters that cannot be overwritten by other tasks — forgetting is structurally prevented rather than penalised.

**Variants:**
- **Progressive Neural Networks:** add a new column of layers for each new task; lateral connections to old columns allow knowledge transfer but old weights are frozen
- **PackNet:** after training on T1, prune the least-used weights; assign freed capacity to T2; freeze T1 weights permanently
- **Piggyback / HAT:** learn a binary mask over shared weights per task; each task activates a different subset of the network
- **LoRA / adapter-based:** attach small adapter modules per task; backbone frozen

**Trade-off:** model size grows with number of tasks (expansion methods), or capacity is shared and eventually exhausted (mask methods).

---

## Summary Table

| Family | Mechanism | Forgetting Prevention | Cost |
|--------|-----------|----------------------|------|
| Replay | Store/generate old data, mix with new | Old-task gradients alongside new | Buffer memory / generation cost |
| Regularisation | Penalise changes to important weights | Constrain weight movement | Approximation errors; capacity saturation |
| Architecture | Separate parameters per task | Structural isolation | Growing model size |

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)

# =============================================================
# Demonstrate all 3 families on the same 2-task problem
# Task 1 and Task 2 are binary classification in different regions
# =============================================================

def make_task(n=300, cx=0.0, cy=0.0):
    X = np.vstack([
        np.random.randn(n//2, 2) + [cx,   cy],
        np.random.randn(n//2, 2) + [cx+3, cy]
    ]).astype('float32')
    y = np.array([0]*(n//2) + [1]*(n//2), dtype='int64')
    return torch.FloatTensor(X), torch.LongTensor(y)

X1, y1 = make_task(cx=0.0,  cy=0.0)   # Task 1: left region
X2, y2 = make_task(cx=8.0,  cy=0.0)   # Task 2: right region

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 2)
        )
    def forward(self, x): return self.net(x)

def train_net(model, opt, X, y, n=300):
    model.train()
    for _ in range(n):
        loss = nn.CrossEntropyLoss()(model(X), y)
        opt.zero_grad(); loss.backward(); opt.step()

def acc(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()


# ── 1. NAIVE SEQUENTIAL (baseline — catastrophic forgetting) ──────────────
naive = Net(); opt_n = optim.Adam(naive.parameters(), lr=1e-3)
train_net(naive, opt_n, X1, y1, 300)
t1_before = acc(naive, X1, y1)
train_net(naive, opt_n, X2, y2, 300)
t1_naive = acc(naive, X1, y1)
t2_naive = acc(naive, X2, y2)


# ── 2. REPLAY ─────────────────────────────────────────────────────────────
replay = Net(); opt_r = optim.Adam(replay.parameters(), lr=1e-3)
train_net(replay, opt_r, X1, y1, 300)
# Buffer: 80 samples from T1
idx = np.random.choice(len(X1), 80, replace=False)
bX, by = X1[idx], y1[idx]
for _ in range(300):
    mX = torch.cat([X2[:100], bX])
    my = torch.cat([y2[:100], by])
    p  = torch.randperm(len(mX))
    nn.CrossEntropyLoss()(replay(mX[p]), my[p]).backward()
    opt_r.step(); opt_r.zero_grad()
t1_replay = acc(replay, X1, y1)
t2_replay = acc(replay, X2, y2)


# ── 3. REGULARISATION (EWC-style) ─────────────────────────────────────────
ewc = Net(); opt_e = optim.Adam(ewc.parameters(), lr=1e-3)
train_net(ewc, opt_e, X1, y1, 300)

# Compute Fisher importance
fisher = {}
for name, param in ewc.named_parameters():
    fisher[name] = torch.zeros_like(param)

ewc.eval()
for xi, yi in zip(X1[:100], y1[:100]):
    ewc.zero_grad()
    out  = ewc(xi.unsqueeze(0))
    loss = nn.CrossEntropyLoss()(out, yi.unsqueeze(0))
    loss.backward()
    for name, param in ewc.named_parameters():
        if param.grad is not None:
            fisher[name] += param.grad.data ** 2 / 100

theta_star = {n: p.data.clone() for n, p in ewc.named_parameters()}

LAMBDA = 5000
for _ in range(300):
    ewc.train()
    task_loss = nn.CrossEntropyLoss()(ewc(X2), y2)
    ewc_loss  = sum(
        (fisher[n] * (p - theta_star[n]) ** 2).sum()
        for n, p in ewc.named_parameters()
    )
    loss = task_loss + LAMBDA * ewc_loss
    opt_e.zero_grad(); loss.backward(); opt_e.step()

t1_ewc = acc(ewc, X1, y1)
t2_ewc = acc(ewc, X2, y2)


# ── 4. ARCHITECTURE (Progressive Network simplified) ──────────────────────
class ProgressiveNet(nn.Module):
    """Column per task; old weights frozen after T1."""
    def __init__(self):
        super().__init__()
        self.col1 = nn.Sequential(nn.Linear(2,64), nn.ReLU(), nn.Linear(64,2))
        self.col2 = nn.Sequential(nn.Linear(2,64), nn.ReLU(), nn.Linear(64,2))
    def forward_task1(self, x): return self.col1(x)
    def forward_task2(self, x): return self.col2(x)

prog = ProgressiveNet(); opt_p = optim.Adam(prog.col1.parameters(), lr=1e-3)
for _ in range(300):
    loss = nn.CrossEntropyLoss()(prog.forward_task1(X1), y1)
    opt_p.zero_grad(); loss.backward(); opt_p.step()

# Freeze col1, train only col2 on T2
for p in prog.col1.parameters(): p.requires_grad_(False)
opt_p2 = optim.Adam(prog.col2.parameters(), lr=1e-3)
for _ in range(300):
    loss = nn.CrossEntropyLoss()(prog.forward_task2(X2), y2)
    opt_p2.zero_grad(); loss.backward(); opt_p2.step()

t1_prog = acc(prog, X1, y1)  # col1 never changed!
t2_prog = acc(prog, X2, y2)  # col2 for T2


# ── RESULTS ───────────────────────────────────────────────────────────────
print('=== Continual Learning Comparison ===')
print(f'{'Method':<25} {'T1 acc after T2':>18} {'T2 acc':>10}')
print('-' * 55)
print(f'{'Naive (forgetting)':<25} {t1_naive:>17.1%} {t2_naive:>9.1%}')
print(f'{'Replay (ER buffer)':<25} {t1_replay:>17.1%} {t2_replay:>9.1%}')
print(f'{'Regularisation (EWC)':<25} {t1_ewc:>17.1%} {t2_ewc:>9.1%}')
print(f'{'Architecture (ProgNet)':<25} {t1_prog:>17.1%} {t2_prog:>9.1%}')

In [ ]:
# =============================================================
# Bar chart comparison
# =============================================================
methods = ['Naive\n(forgetting)', 'Replay\n(ER)', 'Regularisation\n(EWC)', 'Architecture\n(ProgNet)']
t1_accs = [t1_naive, t1_replay, t1_ewc, t1_prog]
t2_accs = [t2_naive, t2_replay, t2_ewc, t2_prog]

x = np.arange(len(methods))
width = 0.35
cols_t1 = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']
cols_t2 = [c + '99' if isinstance(c, str) else c for c in cols_t1]  # lighter

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, t1_accs, width, label='Task 1 acc (after T2 training)',
               color=['#c0392b','#27ae60','#2980b9','#8e44ad'], alpha=0.9)
bars2 = ax.bar(x + width/2, t2_accs, width, label='Task 2 acc',
               color=['#e74c3c','#2ecc71','#3498db','#9b59b6'], alpha=0.6)

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f'{h:.0%}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(methods, fontsize=10)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.12)
ax.set_title('Three Continual Learning Families vs Naive Sequential Training\n'
             'All three recover Task 1 knowledge that naive training forgets')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4, linewidth=1)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================
# Visualisation: EWC Fisher importance — which weights matter?
# =============================================================
import torch

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

for ax, (name, f_tensor) in zip(axes,
        {k: v for k, v in fisher.items() if 'weight' in k}.items()):
    F = f_tensor.detach().numpy()
    if F.ndim == 1:
        F = F.reshape(1, -1)
    im = ax.imshow(F, cmap='hot', aspect='auto')
    ax.set_title(f'Fisher importance\n{name}\n(bright = important for T1)', fontsize=8)
    plt.colorbar(im, ax=ax)

fig.suptitle('EWC: Fisher Information Matrix — identifies which weights to protect',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print('EWC interpretation:')
print('  High Fisher value → weight was important for Task 1 loss')
print('  EWC adds penalty: cost grows if this weight moves away from its T1 value')
print('  Result: model is forced to learn T2 using DIFFERENT weights')
print('  Those T1-important weights are effectively frozen')

## Exam Quick-Reference

**Replay methods:** Store (or generate) past-task samples and mix them with new training batches. Old-task gradients counteract new-task forgetting by continuously rehearsing previous knowledge.

**Regularisation-based methods:** Add a penalty term to the loss that discourages large changes to weights that were important for old tasks (e.g., EWC uses Fisher information to identify and protect important weights). The model learns new tasks within the constraints set by old tasks.

**Architecture-based methods:** Assign separate model components (subnetworks, frozen columns, task-specific masks/adapters) to different tasks. Since tasks use different parameters, there is no interference by design — forgetting is structurally impossible for frozen components.

**One mark each** — one sentence on core principle + one sentence on how it prevents forgetting.